# Homework: Tuples Comprehensions | Product Photo Preparation

## Before you begin
1. In Colab, choose File > Save a copy in Drive.
2. Run the checking setup cell, then work from top to bottom.
3. Replace each `...` with your code or answer. Keep the supplied variable and function names.
4. Run each `grader.check(...)` cell for feedback. A passing check covers the supplied cases; also check your reasoning and the requested programming approach.
5. Save your work regularly. At the end, run the final total and receipt cell, download the .ipynb file, and upload it to Moodle.

All retail records in this homework are fictional teaching data, not records from any other company.


In [ ]:
#@title Run this cell first -- homework checking setup { display-mode: "form" }
#@markdown Run this cell before anything else, every time you open the notebook.
#@markdown <br><br>It sets up `grader.check(...)` so you can check your answers as you go.
#@markdown Nothing is installed and nothing is downloaded -- this notebook is
#@markdown self-contained, so it works even if Colab or GitHub is having a bad day.

import base64
import datetime
import doctest
import json
import sys

RECEIPT_PREFIX = "CS1910-RECEIPT:"

_HTML = """<div style="font-family:system-ui,sans-serif;font-size:14px;margin:4px 0;
 padding:10px 12px;border-left:5px solid {bar};background:{bg};border-radius:4px">
 <b>{name}</b> &mdash; {earned}/{possible} {verdict}{detail}</div>"""


class Result:
    """One question's outcome. Displays as HTML in Colab, as text everywhere else.

    The text form is what lands in the notebook's saved output and what the grader
    parses, so keep the first line machine-readable.
    """

    def __init__(self, name, earned, possible, messages):
        self.name, self.earned, self.possible = name, earned, possible
        self.messages = messages

    @property
    def passed(self):
        return self.earned == self.possible

    def __repr__(self):
        head = "{}: {}/{} {}".format(self.name, self.earned, self.possible,
                                     "PASSED" if self.passed else "FAILED")
        return "\n".join([head] + self.messages)

    def _repr_html_(self):
        bar, bg = ("#2e7d32", "#edf7ed") if self.passed else ("#c62828", "#fdecea")
        verdict = "passed" if self.passed else "not passing yet"
        detail = ""
        if self.messages:
            body = "\n".join(self.messages).replace("&", "&amp;").replace("<", "&lt;")
            detail = ('<pre style="margin:8px 0 0;white-space:pre-wrap;font-size:12px;'
                      'color:#444">{}</pre>'.format(body))
        return _HTML.format(bar=bar, bg=bg, name=self.name, earned=self.earned,
                            possible=self.possible, verdict=verdict, detail=detail)


def run_case(code, globs):
    """Run one doctest string. Returns (passed, report_text).

    `get_doctest` copies `globs`, so a case that rebinds a name cannot leak that
    change back into the student's namespace or into the next case.
    """
    parser = doctest.DocTestParser()
    test = parser.get_doctest(code.rstrip() + "\n", dict(globs), "case", None, 0)
    runner = doctest.DocTestRunner(verbose=False,
                                   optionflags=doctest.NORMALIZE_WHITESPACE)
    out = []
    try:
        runner.run(test, out=out.append, clear_globs=False)
    except Exception as exc:                      # a case that raises is a failure
        return False, "{}: {}".format(type(exc).__name__, exc)
    return runner.failures == 0, "".join(out)


def score(name, spec, globs):
    """Score one question against `globs`. Returns a Result."""
    cases = spec["suites"][0]["cases"] if "suites" in spec else spec["cases"]
    per_case = bool(cases) and all("points" in c for c in cases)

    earned = possible = 0
    failures, notes = [], []
    for case in cases:
        ok, report = run_case(case["code"], globs)
        if per_case:
            possible += case["points"]
            earned += case["points"] if ok else 0
        if ok:
            if case.get("success_message"):
                notes.append(case["success_message"])
        else:
            failures.append(case.get("failure_message") or _trim(report))

    if not per_case:
        possible = spec.get("points", len(cases))
        earned = 0 if failures else possible

    return Result(name, earned, possible, failures or notes)


def _trim(report, limit=600):
    """Doctest reports are verbose; keep the part a first-year student can act on."""
    text = report.strip()
    return text if len(text) <= limit else text[:limit] + "\n... (truncated)"


def caller_namespace():
    """Where the student's variables live.

    In Colab and Jupyter that is the interactive namespace, which is right even when
    `check()` is called from inside a function. Outside IPython -- the headless grader,
    or a plain `python` run -- fall back to the calling frame's globals, which is the
    namespace `exec` was handed.
    """
    try:
        shell = get_ipython()           # noqa: F821  (injected by IPython)
        if shell is not None:
            return shell.user_ns
    except NameError:
        pass
    return sys._getframe(2).f_globals


class Grader:
    """Holds the tests for one homework and remembers what has been checked.

    Named `grader` in the student notebook so existing `grader.check("q1")` cells
    keep working untouched.
    """

    def __init__(self, hw, tests):
        self.hw, self.tests, self.log = hw, tests, {}

    def check(self, name, globs=None):
        if name not in self.tests:
            raise KeyError("no test named {!r} in {} (have: {})".format(
                name, self.hw, ", ".join(sorted(self.tests))))
        result = score(name, self.tests[name], globs or caller_namespace())
        self.log[name] = (result.earned, result.possible)
        return result

    def check_all(self, globs):
        """Score every question against one namespace. This is what grading uses."""
        results = [score(n, self.tests[n], globs) for n in sorted(self.tests)]
        self.log = {r.name: (r.earned, r.possible) for r in results}
        return results

    def summary(self):
        return {"hw": self.hw,
                "results": {k: list(v) for k, v in sorted(self.log.items())},
                "total": sum(e for e, _ in self.log.values()),
                "possible": sum(p for _, p in self.log.values())}

    def receipt(self, student_id="", extra=None):
        """Print the one-line machine-readable receipt and return it.

        The grader re-runs the tests itself, so this line is a convenience for the
        student and a fast triage signal -- never the authoritative score.
        """
        body = self.summary()
        body["id"] = str(student_id)
        body["t"] = datetime.datetime.now().isoformat(timespec="seconds")
        if extra:
            body.update(extra)
        blob = base64.b64encode(
            json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
        ).decode()
        line = RECEIPT_PREFIX + blob
        print(line)
        return line


def parse_receipt(text):
    """Pull the last receipt out of arbitrary text. Returns a dict, or None."""
    found = None
    for chunk in text.split(RECEIPT_PREFIX)[1:]:
        token = chunk.split()[0] if chunk.split() else ""
        try:
            found = json.loads(base64.b64decode(token))
        except Exception:
            continue
    return found


def load_ok_tests(directory):
    """Read a directory of `q*.py` test files (OK_FORMAT doctests) into {name: spec}."""
    import pathlib
    tests = {}
    for path in sorted(pathlib.Path(directory).glob("*.py")):
        namespace = {}
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)
        spec = namespace["test"]
        tests[spec["name"]] = spec
    return tests

_TESTS = {'q1': {'cases': [{'code': ">>> q1_option\n'B'", 'points': 1}], 'name': 'q1', 'points': 1},
 'q2': {'cases': [{'code': '>>> grayscale((10, 20, 30))\n(20, 20, 20)', 'points': 1},
                  {'code': '>>> grayscale((1, 1, 0))\n(0, 0, 0)', 'points': 1},
                  {'code': '>>> grayscale((255, 255, 255))\n(255, 255, 255)', 'points': 1}],
        'name': 'q2',
        'points': 3},
 'q3': {'cases': [{'code': '>>> a = [(9, 0, 0), (30, 60, 90)]\n'
                           '>>> grayscale_row(a) == [(3, 3, 3), (60, 60, 60)] and a == [(9, 0, '
                           '0), (30, 60, 90)]\n'
                           'True',
                   'points': 1},
                  {'code': '>>> grayscale_row([])\n[]', 'points': 1}],
        'name': 'q3',
        'points': 2},
 'q4': {'cases': [{'code': '>>> a = [[(9, 0, 0)], [(0, 6, 0)]]\n'
                           '>>> b = grayscale_image(a)\n'
                           '>>> b == [[(3, 3, 3)], [(2, 2, 2)]] and a == [[(9, 0, 0)], [(0, 6, '
                           '0)]] and b[0] is not a[0]\n'
                           'True',
                   'points': 1},
                  {'code': '>>> grayscale_image([[], []])\n[[], []]', 'points': 1},
                  {'code': '>>> grayscale_image([])\n[]', 'points': 1}],
        'name': 'q4',
        'points': 3},
 'q5': {'cases': [{'code': '>>> brighten_pixel((250, 10, 0), 20)\n(255, 30, 20)', 'points': 1},
                  {'code': '>>> brighten_pixel((5, 20, 255), -30)\n(0, 0, 225)', 'points': 1},
                  {'code': '>>> a = [[(1, 2, 3)]]\n'
                           '>>> b = brighten_image(a, 2)\n'
                           '>>> b == [[(3, 4, 5)]] and a == [[(1, 2, 3)]] and b[0] is not '
                           'a[0]\n'
                           'True',
                   'points': 1},
                  {'code': '>>> brighten_image([], 20)\n[]', 'points': 1}],
        'name': 'q5',
        'points': 4}}

grader = Grader('11-Tuples-Comprehensions', _TESTS)
check = grader.check

print("Ready. 11-Tuples-Comprehensions loaded: 5 question(s), 13 point(s).")
print("Check a question any time with, for example:  grader.check(\"q1\")")

## Question 1: Understand a pixel record

In this simplified 8-bit RGB format, a pixel is a tuple of three integers from 0 to 255. What happens after `pixel = (255, 0, 0)` when Python executes `pixel[0] = 10`? Set `q1_option` to `"A"` (the tuple changes), `"B"` (a TypeError), or `"C"` (a new list). Tuples are our design choice for fixed records, not a requirement of all image software.

Points: 1

In [ ]:
q1_option = ...

In [ ]:
grader.check("q1")

## Question 2: Convert one product pixel

Write `grayscale(pixel)`. Unpack `(red, green, blue)` and return a new tuple with all channels equal to the integer average `(red + green + blue) // 3`. This is a simple teaching filter, not a perceptually weighted greyscale conversion.

Points: 3

In [ ]:
def grayscale(pixel):
    ...

In [ ]:
grader.check("q2")

## Question 3: Process one row with a comprehension

Write `grayscale_row(row)` using one list comprehension calling your `grayscale` function. Return a new list of pixel tuples. For example, `[(9, 0, 0), (30, 60, 90)]` becomes `[(3, 3, 3), (60, 60, 60)]`. An empty row gives an empty list. Leave the input unchanged.

Points: 2

In [ ]:
def grayscale_row(row):
    ...

In [ ]:
grader.check("q3")

## Question 4: Prepare a complete thumbnail

An image is a list of rows, each containing pixel tuples. Write `grayscale_image(image)` using ordinary nested loops. Call `grayscale` for each pixel and create new lists for the image and its rows. Leave the original unchanged. This practices the same operation at two levels before the final comprehension task.

Points: 3

In [ ]:
def grayscale_image(image):
    ...

In [ ]:
grader.check("q4")

## Question 5: Adjust brightness safely

Write `brighten_pixel(pixel, amount)` to return a tuple with `amount` added to each channel, limited to the range 0 through 255. Negative amounts darken the photo. Then write `brighten_image(image, amount)` using a nested list comprehension calling that helper. Return a new image and leave the original unchanged. Example: `(250, 10, 0)` with amount 20 becomes `(255, 30, 20)`.

Points: 4

In [ ]:
def brighten_pixel(pixel, amount):
    ...

def brighten_image(image, amount):
    ...

In [ ]:
grader.check("q5")

## Beyond the homework
Optional, not graded: describe one additional edge case and explain what the correct result should be. Test it in a separate scratch cell without changing the supplied data above.

# Submission
- Run all cells in order, including the final total and receipt cell below.
- Save your completed notebook.
- Choose File > Download > Download .ipynb and upload that file to the appropriate Moodle homework submission.
- Submit the notebook, not a screenshot or receipt alone. Your instructor re-runs your code to determine your mark.


In [ ]:
#@title Finished? Run this to see your total { display-mode: "form" }
#@markdown Re-checks every question and prints your score, plus a receipt line.
#@markdown <br><br>Your grade comes from your instructor re-running this notebook,
#@markdown so the receipt is for your records -- make sure the notebook itself is saved.

student_id = ""  #@param {type:"string"}

for _r in grader.check_all(globals()):
    print(_r)
_s = grader.summary()
print("\nTOTAL: {}/{}".format(_s["total"], _s["possible"]))
grader.receipt(student_id)
